In [1]:
import math
import collections
from typing import List, Dict, Tuple, Set

# =====================================================================
# PHASE 1: PRE-TOKENIZER & TOKENS SEED STRUCTURE
# =====================================================================

class MinGramPPTokenizer:
    def __init__(self):
        # Maps token string -> log probability (float)
        self.vocab: Dict[str, float] = {}
        # Reverse mapping for decoding: token_id -> token string
        self.id_to_token: List[str] = []
        self.token_to_id: Dict[str, int] = {}

        # Trie root for fast prefix matching during lattice construction
        self.trie = {}
        # Set to track guaranteed atomic characters
        self.atomics: Set[str] = set()

    def _add_to_trie(self, token: str):
        """Inserts a token string into the character prefix trie."""
        node = self.trie
        for char in token:
            if char not in node:
                node[char] = {}
            node = node[char]
        node['<EOS>'] = True  # Marks a valid vocabulary token endpoint

    def load_seed_vocabulary(self, candidate_counts: Dict[str, int], corpus_text: str):
        """
        Initializes the vocabulary using overshot BPE tokens or raw frequencies.
        Guarantees all unique single characters are registered as atomic tokens.
        """
        # 1. Identify and extract all unique characters to serve as atomic safety nets
        self.atomics = set(corpus_text)

        # 2. Accumulate all unique candidate tokens
        all_tokens = set(candidate_counts.keys()) | self.atomics

        # Calculate base total frequency for probability normalization
        total_count = sum(candidate_counts.get(t, 0) for t in all_tokens) + len(self.atomics)

        self.vocab = {}
        self.trie = {}

        for token in all_tokens:
            # Give missing atomic tokens a baseline count of 1 to avoid log(0)
            count = candidate_counts.get(token, 0)
            if token in self.atomics and count == 0:
                count = 1

            # Compute log probability: log(p)
            self.vocab[token] = math.log(count / total_count)
            self._add_to_trie(token)

        # Rebuild ID maps
        self.id_to_token = sorted(list(self.vocab.keys()))
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

    # =====================================================================
    # PHASE 2: LATTICE CONSTRUCTION & VITERBI DECODER
    # =====================================================================

    def _find_matches(self, text: str, start_idx: int) -> List[Tuple[str, int]]:
        """
        Uses the Trie to find all vocabulary tokens matching text starting at start_idx.
        Returns a list of tuples: (matched_token_string, token_length)
        """
        matches = []
        node = self.trie
        curr_idx = start_idx

        while curr_idx < len(text):
            char = text[curr_idx]
            if char not in node:
                break
            node = node[char]
            curr_idx += 1
            if '<EOS>' in node:
                matched_token = text[start_idx:curr_idx]
                matches.append((matched_token, curr_idx - start_idx))

        # Defensive fallback: If no vocabulary token matches, force the atomic character
        if not matches and start_idx < len(text):
            atomic_char = text[start_idx]
            if atomic_char in self.vocab:
                matches.append((atomic_char, 1))

        return matches

    def tokenize(self, text: str) -> List[str]:
        """
        MinGram-PP Viterbi Decoder.
        Finds the path minimizing token count (|s|), breaking ties with max joint log-prob.
        """
        if not text:
            return []

        n = len(text)
        # dp[i] tracks: (min_tokens_count, max_joint_log_prob, backpointer_index, matched_token)
        # Initialize with infinity for token counts and negative infinity for probabilities
        dp = [(float('inf'), float('-inf'), -1, "")] * (n + 1)
        dp[0] = (0, 0.0, -1, "") # Base case at text start

        # Delta weight to make log-prob strictly act as a tie-breaker
        # An extra token must always cost more than any possible log-prob gain.
        delta = 1e-7

        # Forward dynamic programming pass
        for i in range(n):
            if dp[i][0] == float('inf'):
                continue # Unreachable state

            # Query the Trie for all valid tokens starting at position i
            matches = self._find_matches(text, i)

            for token, length in matches:
                next_idx = i + length
                token_score = self.vocab[token]

                # MinGram Objective Function cost transition:
                # cost = current_tokens + 1 - (delta * token_log_prob)
                potential_tokens = dp[i][0] + 1
                potential_prob = dp[i][1] + token_score

                # Evaluation step
                if potential_tokens < dp[next_idx][0]:
                    # Found a path with strictly fewer tokens
                    dp[next_idx] = (potential_tokens, potential_prob, i, token)
                elif potential_tokens == dp[next_idx][0]:
                    # Token count tie: choose the path maximizing log-probability
                    if potential_prob > dp[next_idx][1]:
                        dp[next_idx] = (potential_tokens, potential_prob, i, token)

        # Backward reconstruction pass
        tokens = []
        curr = n
        while curr > 0:
            prev_idx = dp[curr][2]
            token = dp[curr][3]
            tokens.append(token)
            curr = prev_idx

        return tokens[::-1] # Reverse to restore normal reading order

In [2]:
# Instantiate the tokenizer
tokenizer = MinGramPPTokenizer()

# Let's mock a corpus and an overshot seed vocabulary list
mock_corpus = "higher higher learning high street"
mock_seed_counts = {
    "h": 10, "i": 10, "g": 10, "h": 10, "e": 10, "r": 10,
    "high": 5,
    "higher": 15,
    "learning": 2,
    "street": 3
}

# Load the seed tokens into the pipeline
tokenizer.load_seed_vocabulary(mock_seed_counts, mock_corpus)

# Test cases evaluating the inference decoder
print("--- TESTING DECODER INFERENCE ---")
print("Input: 'higher'   -> Decoded:", tokenizer.tokenize("higher"))
print("Input: 'high'     -> Decoded:", tokenizer.tokenize("high"))
print("Input: 'highest'  -> Decoded:", tokenizer.tokenize("highest"))
# 'highest' contains 'high' + atomic characters 'e', 's', 't'

--- TESTING DECODER INFERENCE ---
Input: 'higher'   -> Decoded: ['higher']
Input: 'high'     -> Decoded: ['high']
Input: 'highest'  -> Decoded: ['high', 'e', 's', 't']


In [4]:
import numpy as np

# Let's add the EM and Pruning methods to our tokenizer class.
# We will inject these methods dynamically so you don't have to redefine the whole class.

def fit_em_step(self, corpus_sequences: List[str]):
    """
    PHASE 3: Hard-EM Optimization Step
    Decodes the corpus using the current Viterbi lattice decoder,
    accumulates counts along the winning paths, and updates log-probabilities.
    """
    token_counts = collections.Counter()

    # 1. E-Step: Tokenize the text sequences and count chosen tokens
    for text in corpus_sequences:
        if not text:
            continue
        chosen_tokens = self.tokenize(text)
        for token in chosen_tokens:
            token_counts[token] += 1

    # 2. M-Step: Re-calculate log probabilities based on winning paths
    # Always keep atomic characters safe by giving them a baseline count if they dropped to 0
    all_tokens = set(self.vocab.keys())
    total_count = sum(token_counts.values()) + len(self.atomics)

    new_vocab = {}
    new_trie = {}

    # We clear and rebuild the trie strictly for active tokens + atomics
    self.trie = {}

    for token in all_tokens:
        count = token_counts[token]
        if token in self.atomics and count == 0:
            count = 1  # Safety floor for atomic characters

        if count > 0:
            new_vocab[token] = math.log(count / total_count)
            self._add_to_trie(token)
        else:
            # Token was completely unused in this EM iteration.
            # We give it a heavily penalized score rather than dropping it immediately,
            # allowing it to be naturally weeded out during the pruning phase.
            new_vocab[token] = float('-inf')

    self.vocab = new_vocab
    self.id_to_token = sorted(list(self.vocab.keys()))
    self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

def train_prune(self, corpus_sequences: List[str], target_vocab_size: int, prune_factor: float = 0.2, em_steps_per_iter: int = 2):
    """
    PHASE 4: PathPiece-style Iterative Pruning (-PP)
    Gradually shrinks the vocabulary down to target_vocab_size by executing
    EM loops, sorting by score, and trimming the least valuable tokens iteratively.
    """
    print(f"Starting training loop. Initial Seed Vocab Size: {len(self.vocab)}")

    while len(self.vocab) > target_vocab_size:
        # Run a few steps of Hard-EM to stabilize token usage counts on this subset
        for em_round in range(em_steps_per_iter):
            self.fit_em_step(corpus_sequences)

        # Calculate how many tokens we want to drop in this specific step
        current_size = len(self.vocab)
        num_to_drop = int(current_size * prune_factor)

        # Guardrail: Make sure we don't accidentally drop below our target vocabulary size
        if current_size - num_to_drop < target_vocab_size:
            num_to_drop = current_size - target_vocab_size

        if num_to_drop <= 0:
            break

        # Sort tokens by their log-probability score
        # Crucial Rule: Non-atomic tokens are candidates for removal; atomic tokens are protected
        non_atomic_tokens = [t for t in self.vocab.keys() if t not in self.atomics]

        # Sort non-atomics ascending by score (lowest probability first)
        non_atomic_tokens.sort(key=lambda t: self.vocab[t])

        # Identify the tokens to evict
        tokens_to_evict = set(non_atomic_tokens[:num_to_drop])

        # Build the pruned vocabulary map
        pruned_vocab = {}
        self.trie = {}

        for token, score in self.vocab.items():
            if token not in tokens_to_evict:
                pruned_vocab[token] = score
                self._add_to_trie(token)

        self.vocab = pruned_vocab
        self.id_to_token = sorted(list(self.vocab.keys()))
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

        print(f" Pruned {num_to_drop} tokens. Current Vocab Size: {len(self.vocab)}")

    # Final cleanup: One final EM round to lock in true probabilities for remaining entries
    self.fit_em_step(corpus_sequences)
    print(f"Training Complete! Final Vocab Size: {len(self.vocab)}")

# Attach the functions to our existing class blueprint
MinGramPPTokenizer.fit_em_step = fit_em_step
MinGramPPTokenizer.train_prune = train_prune

In [5]:
# 1. Setup sample corpus text and an intentionally overblown mock BPE seed vocabulary
sample_corpus = [
    "higher higher learning",
    "high street fashion",
    "learning tokens from scratch",
    "inference engineering optimization"
]

# Flatten corpus to extract unique atomic characters safely
full_text = " ".join(sample_corpus)

# Overshot seed tokens (like what an initial loose BPE execution would yield)
overshot_seed_counts = {
    "high": 10, "higher": 8, "learning": 5, "street": 4, "fashion": 3,
    "tokens": 4, "scratch": 2, "inference": 6, "engineering": 5, "optimization": 5,
    "low": 1, "unrelated": 1, "garbage": 1, "extra": 1, "random": 1
}

# 2. Instantiate and load seed tokens
pp_tokenizer = MinGramPPTokenizer()
pp_tokenizer.load_seed_vocabulary(overshot_seed_counts, full_text)

# 3. Target an exact final vocabulary size of 32 (including unique atomic characters)
pp_tokenizer.train_prune(sample_corpus, target_vocab_size=32, prune_factor=0.25)

# 4. Test Tokenization Output post-pruning
print("\n--- FINAL PRODUCTION TOKENS CHECK ---")
print("Text: 'inference optimization' ->", pp_tokenizer.tokenize("inference optimization"))
print("Text: 'higher fashion'          ->", pp_tokenizer.tokenize("higher fashion"))

Starting training loop. Initial Seed Vocab Size: 33
 Pruned 1 tokens. Current Vocab Size: 32
Training Complete! Final Vocab Size: 32

--- FINAL PRODUCTION TOKENS CHECK ---
Text: 'inference optimization' -> ['inference', ' ', 'optimization']
Text: 'higher fashion'          -> ['higher', ' ', 'fashion']


In [6]:
import re

def tokenize_optimized(self, text: str) -> List[str]:
    """
    An production-minded inference wrapper.
    Splits the string into words/chunks first using regex to ensure the
    Viterbi Dynamic Programming array never gets bottlenecked by massive strings.
    """
    if not text:
        return []

    # Simple pre-tokenization: splits by spaces but keeps the spaces attached
    # to the front of words, mimicking the standard ' ' meta-character layout.
    # Pattern groups spaces with subsequent letters, or isolates punctuation.
    chunks = re.findall(r'\s*\w+|\s+|[^\w\s]', text)

    final_tokens = []
    for chunk in chunks:
        final_tokens.extend(self._tokenize_chunk(chunk))
    return final_tokens

def _tokenize_chunk(self, chunk: str) -> List[str]:
    """Internal Viterbi solver for short, isolated text chunks."""
    n = len(chunk)
    dp = [(float('inf'), float('-inf'), -1, "")] * (n + 1)
    dp[0] = (0, 0.0, -1, "")
    delta = 1e-7

    for i in range(n):
        if dp[i][0] == float('inf'):
            continue

        matches = self._find_matches(chunk, i)
        for token, length in matches:
            next_idx = i + length
            token_score = self.vocab[token]

            potential_tokens = dp[i][0] + 1
            potential_prob = dp[i][1] + token_score

            if potential_tokens < dp[next_idx][0]:
                dp[next_idx] = (potential_tokens, potential_prob, i, token)
            elif potential_tokens == dp[next_idx][0]:
                if potential_prob > dp[next_idx][1]:
                    dp[next_idx] = (potential_tokens, potential_prob, i, token)

    tokens = []
    curr = n
    while curr > 0:
        prev_idx = dp[curr][2]
        token = dp[curr][3]
        tokens.append(token)
        curr = prev_idx

    return tokens[::-1]

# Overwrite / attach the optimized approaches
MinGramPPTokenizer._tokenize_chunk = _tokenize_chunk
MinGramPPTokenizer.tokenize = tokenize_optimized

In [7]:
print("--- PRE-TOKENIZED INFERENCE TEST ---")
test_sentence = "testing inference speed optimization framework"
print("Decoded Chunks Path:", pp_tokenizer.tokenize(test_sentence))

--- PRE-TOKENIZED INFERENCE TEST ---
Decoded Chunks Path: ['t', 'e', 's', 't', 'i', 'n', 'g', ' ', 'inference', '', ' ', 'optimization', '']


In [8]:
def encode(self, text: str) -> List[int]:
    """Converts a raw string into a list of integer token IDs."""
    tokens = self.tokenize(text)
    return [self.token_to_id[t] for t in tokens if t in self.token_to_id]

def decode(self, ids: List[int]) -> str:
    """Converts a list of token IDs back into a human-readable string."""
    return "".join([self.id_to_token[i] for i in ids])

# Attach these final inference methods
MinGramPPTokenizer.encode = encode
MinGramPPTokenizer.decode = decode

In [9]:
# 1. Provide a more diverse corpus
real_corpus = [
    "deep learning and inference engineering require absolute optimization",
    "tokenizers are the foundation of large language models",
    "speed and efficiency matter during tokenization inference steps",
    "building an optimized tokenizer framework from scratch using python"
]
combined_text = " ".join(real_corpus)

# 2. Simulate a BPE overshot candidate generator by extracting common substrings
# (In production, you'd run a fast BPE script to target ~1.15x vocabulary size)
from itertools import combinations
substring_counts = collections.Counter()
for sentence in real_corpus:
    words = sentence.split()
    for word in words:
        # Mine all possible sub-strings up to length 7 as candidates
        for i, j in combinations(range(len(word) + 1), 2):
            sub = word[i:j]
            if len(sub) > 1:
                substring_counts[sub] += 2

# Add explicit space handling candidates
substring_counts[" "] = 20
substring_counts["  "] = 5

# 3. Load the candidate pool into MinGram-PP
production_tok = MinGramPPTokenizer()
production_tok.load_seed_vocabulary(substring_counts, combined_text)

# 4. Run the Iterative Pruning Pipeline down to a concise vocabulary of 60 tokens
production_tok.train_prune(real_corpus, target_vocab_size=60, prune_factor=0.2)

print("\n" + "="*50)
print("--- PRODUCTION ROUND-TRIP PERFORMANCE ---")
test_prompt = "deep learning optimization framework"

# Encode to IDs
encoded_ids = production_tok.encode(test_prompt)
print(f"Input Text:  '{test_prompt}'")
print(f"Encoded IDs: {encoded_ids}")
print(f"Token Path:  {production_tok.tokenize(test_prompt)}")

# Decode back to String
decoded_str = production_tok.decode(encoded_ids)
print(f"Decoded Str: '{decoded_str}'")
print(f"Round-trip Match Success: {test_prompt == decoded_str}")

Starting training loop. Initial Seed Vocab Size: 604
 Pruned 120 tokens. Current Vocab Size: 484
 Pruned 96 tokens. Current Vocab Size: 388
 Pruned 77 tokens. Current Vocab Size: 311
 Pruned 62 tokens. Current Vocab Size: 249
 Pruned 49 tokens. Current Vocab Size: 200
 Pruned 40 tokens. Current Vocab Size: 160
 Pruned 32 tokens. Current Vocab Size: 128
 Pruned 25 tokens. Current Vocab Size: 103
 Pruned 20 tokens. Current Vocab Size: 83
 Pruned 16 tokens. Current Vocab Size: 67
 Pruned 7 tokens. Current Vocab Size: 60
Training Complete! Final Vocab Size: 60

--- PRODUCTION ROUND-TRIP PERFORMANCE ---
Input Text:  'deep learning optimization framework'
Encoded IDs: [11, 0, 31, 0, 38, 0, 19]
Token Path:  ['deep', ' ', 'learning', ' ', 'optimization', ' ', 'framework']
Decoded Str: 'deep learning optimization framework'
Round-trip Match Success: True


In [10]:
import math
import collections
import re
from typing import List, Dict, Tuple, Set

class ByteMinGramPPTokenizer:
    def __init__(self):
        # Vocab maps bytes -> log probability
        self.vocab: Dict[bytes, float] = {}
        self.id_to_token: List[bytes] = []
        self.token_to_id: Dict[bytes, int] = {}
        self.trie = {}
        # Protect all 256 individual possible byte values as atomic safety nets
        self.atomics: Set[bytes] = {bytes([i]) for i in range(256)}

    def _add_to_trie(self, token_bytes: bytes):
        node = self.trie
        for byte in token_bytes:
            if byte not in node:
                node[byte] = {}
            node = node[byte]
        node['<EOS>'] = True

    def load_seed_vocabulary(self, candidate_counts: Dict[bytes, int]):
        """Initializes vocabulary using raw byte tokens, ensuring all 256 basic bytes are protected."""
        all_tokens = set(candidate_counts.keys()) | self.atomics
        total_count = sum(candidate_counts.get(t, 0) for t in all_tokens) + 256

        self.vocab = {}
        self.trie = {}

        for token in all_tokens:
            count = candidate_counts.get(token, 0)
            if token in self.atomics and count == 0:
                count = 1

            self.vocab[token] = math.log(count / total_count)
            self._add_to_trie(token)

        self.id_to_token = sorted(list(self.vocab.keys()))
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

    def _find_matches(self, byte_seq: bytes, start_idx: int) -> List[Tuple[bytes, int]]:
        matches = []
        node = self.trie
        curr_idx = start_idx

        while curr_idx < len(byte_seq):
            b = byte_seq[curr_idx]
            if b not in node:
                break
            node = node[b]
            curr_idx += 1
            if '<EOS>' in node:
                matched_token = byte_seq[start_idx:curr_idx]
                matches.append((matched_token, curr_idx - start_idx))

        if not matches and start_idx < len(byte_seq):
            # Fall back safely to a single individual byte
            single_byte = byte_seq[start_idx:start_idx+1]
            matches.append((single_byte, 1))

        return matches

    def tokenize(self, text: str) -> List[bytes]:
        """Pre-tokenizes text string, converts chunks to bytes, and decodes via Viterbi lattice."""
        if not text:
            return []

        # Regex chunking to prevent large lattice memory bottlenecks
        chunks = re.findall(r'\s*\w+|\s+|[^\w\s]', text)
        final_tokens = []

        for chunk in chunks:
            # Convert text chunk to raw UTF-8 bytes
            chunk_bytes = chunk.encode('utf-8')
            final_tokens.extend(self._tokenize_byte_chunk(chunk_bytes))

        return final_tokens

    def _tokenize_byte_chunk(self, chunk_bytes: bytes) -> List[bytes]:
        n = len(chunk_bytes)
        if n == 0:
            return []

        dp = [(float('inf'), float('-inf'), -1, b"")] * (n + 1)
        dp[0] = (0, 0.0, -1, b"")
        delta = 1e-7

        for i in range(n):
            if dp[i][0] == float('inf'):
                continue

            matches = self._find_matches(chunk_bytes, i)
            for token_bytes, length in matches:
                next_idx = i + length
                token_score = self.vocab[token_bytes]

                potential_tokens = dp[i][0] + 1
                potential_prob = dp[i][1] + token_score

                if potential_tokens < dp[next_idx][0]:
                    dp[next_idx] = (potential_tokens, potential_prob, i, token_bytes)
                elif potential_tokens == dp[next_idx][0]:
                    if potential_prob > dp[next_idx][1]:
                        dp[next_idx] = (potential_tokens, potential_prob, i, token_bytes)

        tokens = []
        curr = n
        while curr > 0:
            prev_idx = dp[curr][2]
            token_bytes = dp[curr][3]
            tokens.append(token_bytes)
            curr = prev_idx

        return tokens[::-1]

    def encode(self, text: str) -> List[int]:
        tokens = self.tokenize(text)
        return [self.token_to_id[t] for t in tokens if t in self.token_to_id]

    def decode(self, ids: List[int]) -> str:
        # Decode array of byte tokens back into a valid string, handling errors gracefully
        byte_stream = b"".join([self.id_to_token[i] for i in ids])
        return byte_stream.decode('utf-8', errors='replace')

In [11]:
# Create a small seed vocabulary mapping common subword bytes
mock_byte_counts = {
    "inference".encode('utf-8'): 10,
    "speed".encode('utf-8'): 8,
    " ".encode('utf-8'): 15
}

byte_tokenizer = ByteMinGramPPTokenizer()
byte_tokenizer.load_seed_vocabulary(mock_byte_counts)

print("--- BYTE FALLBACK TEST ---")
complex_input = "inference speed 🔥"

encoded = byte_tokenizer.encode(complex_input)
decoded = byte_tokenizer.decode(encoded)

print(f"Raw Input:       {complex_input}")
print(f"Generated IDs:   {encoded}")
print(f"Decoded Output:  {decoded}")
print(f"Visualizing Path:{byte_tokenizer.tokenize(complex_input)}")

--- BYTE FALLBACK TEST ---
Raw Input:       inference speed 🔥
Generated IDs:   [106, 32, 117, 32, 242, 161, 150, 167]
Decoded Output:  inference speed 🔥
Visualizing Path:[b'inference', b' ', b'speed', b' ', b'\xf0', b'\x9f', b'\x94', b'\xa5']


In [12]:
class FlatMatrixTrie:
    def __init__(self, vocabulary_list: List[bytes]):
        """
        Builds a highly efficient, cache-friendly Flat Array Trie Matrix.
        Alphabet size is strictly 256 (all possible byte values).
        """
        self.ALPHABET_SIZE = 256

        # Structure to build standard tree states first
        # state 0 is the root node
        states = [{}]
        # Maps state index -> bool indicating if it completes a valid vocabulary token
        self.is_terminal = [False]

        # 1. First build the regular state transitions
        for token_id, token_bytes in enumerate(vocabulary_list):
            curr_state = 0
            for byte in token_bytes:
                if byte not in states[curr_state]:
                    # Create a new state index
                    states[curr_state][byte] = len(states)
                    states.append({})
                    self.is_terminal.append(False)
                curr_state = states[curr_state][byte]
            self.is_terminal[curr_state] = True

        # 2. Flatten the transition graphs into a single contiguous 1D matrix
        num_states = len(states)
        # Initialize flat matrix with -1 (indicating no valid transition)
        self.matrix = [-1] * (num_states * self.ALPHABET_SIZE)

        for state_idx in range(num_states):
            for byte_val, next_state_idx in states[state_idx].items():
                flat_idx = state_idx * self.ALPHABET_SIZE + byte_val
                self.matrix[flat_idx] = next_state_idx

    def find_all_matches(self, byte_seq: bytes, start_idx: int) -> List[int]:
        """
        Traverses the flat matrix using memory offsets.
        Returns a list of token lengths that match valid terminal paths.
        """
        matches = []
        curr_state = 0
        curr_idx = start_idx
        seq_len = len(byte_seq)

        while curr_idx < seq_len:
            byte_val = byte_seq[curr_idx]

            # Direct look-up logic via state offsets: state_idx * 256 + byte_val
            flat_idx = curr_state * self.ALPHABET_SIZE + byte_val
            next_state = self.matrix[flat_idx]

            if next_state == -1:
                break # No further path valid in our vocabulary matrix

            curr_state = next_state
            curr_idx += 1

            if self.is_terminal[curr_state]:
                matches.append(curr_idx - start_idx)

        return matches

In [13]:
# Upgrade our matching method to utilize the new FlatMatrixTrie asset
def upgraded_find_matches(self, byte_seq: bytes, start_idx: int) -> List[Tuple[bytes, int]]:
    # Initialize the Flat Matrix if it hasn't been instantiated yet
    if not hasattr(self, 'flat_trie') or self.flat_trie is None:
        self.flat_trie = FlatMatrixTrie(self.id_to_token)

    lengths = self.flat_trie.find_all_matches(byte_seq, start_idx)
    matches = [(byte_seq[start_idx:start_idx+l], l) for l in lengths]

    if not matches and start_idx < len(byte_seq):
        matches.append((byte_seq[start_idx:start_idx+1], 1))

    return matches

# Bind the flat matrix lookup method
ByteMinGramPPTokenizer._find_matches = upgraded_find_matches
# Reset the internal instance cache to force rebuilding the matrix
byte_tokenizer.flat_trie = None

print("--- TESTING HIGH-PERFORMANCE FLAT MATRIX INFERENCE ---")
encoded_fast = byte_tokenizer.encode(complex_input)
decoded_fast = byte_tokenizer.decode(encoded_fast)

print(f"Flat Matrix Round-trip Result: '{decoded_fast}'")
print(f"Verification Check: {decoded_fast == complex_input}")

--- TESTING HIGH-PERFORMANCE FLAT MATRIX INFERENCE ---
Flat Matrix Round-trip Result: 'inference speed 🔥'
Verification Check: True


In [14]:
def fit_em_step_bytes(self, corpus_sequences: List[str]):
    """
    PHASE 3 (Byte Engine): Hard-EM Optimization Step
    Decodes the corpus sequences into optimized bytes paths, accumulates frequencies,
    and updates scores before invalidating the current flat trie cache.
    """
    token_counts = collections.Counter()

    # E-Step: Tokenize the sequences and accumulate selected path hits
    for text in corpus_sequences:
        chosen_tokens = self.tokenize(text)
        for token in chosen_tokens:
            token_counts[token] += 1

    # M-Step: Recalculate joint log probabilities
    all_tokens = set(self.vocab.keys())
    total_count = sum(token_counts.values()) + len(self.atomics)

    new_vocab = {}
    for token in all_tokens:
        count = token_counts[token]
        if token in self.atomics and count == 0:
            count = 1  # Protect underlying basic byte array

        if count > 0:
            new_vocab[token] = math.log(count / total_count)
        else:
            new_vocab[token] = float('-inf')

    self.vocab = new_vocab
    self.id_to_token = sorted(list(self.vocab.keys()))
    self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

    # Crucial Inference Step: Force the engine to clear and rebuild the
    # High-Performance Flat Matrix Trie on the next tokenization pass!
    self.flat_trie = None

def train_prune_bytes(self, corpus_sequences: List[str], target_vocab_size: int, prune_factor: float = 0.2, em_steps_per_iter: int = 2):
    """
    PHASE 4 (Byte Engine): PathPiece Iterative Pruning Loop
    """
    print(f"Starting Byte Training. Initial Vocabulary Candidates: {len(self.vocab)}")

    while len(self.vocab) > target_vocab_size:
        for _ in range(em_steps_per_iter):
            self.fit_em_step_bytes(corpus_sequences)

        current_size = len(self.vocab)
        num_to_drop = int(current_size * prune_factor)

        if current_size - num_to_drop < target_vocab_size:
            num_to_drop = current_size - target_vocab_size

        if num_to_drop <= 0:
            break

        # Separate out non-atomic byte chunks for pruning consideration
        non_atomic_tokens = [t for t in self.vocab.keys() if t not in self.atomics]
        non_atomic_tokens.sort(key=lambda t: self.vocab[t])

        tokens_to_evict = set(non_atomic_tokens[:num_to_drop])

        pruned_vocab = {}
        for token, score in self.vocab.items():
            if token not in tokens_to_evict:
                pruned_vocab[token] = score

        self.vocab = pruned_vocab
        self.id_to_token = sorted(list(self.vocab.keys()))
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}
        self.flat_trie = None

        print(f" Pruned {num_to_drop} byte tokens. Current Size: {len(self.vocab)}")

    self.fit_em_step_bytes(corpus_sequences)
    print(f"Training Complete! Final Clean Matrix Size: {len(self.vocab)}")

# Attach the byte loops to the class blueprint
ByteMinGramPPTokenizer.fit_em_step_bytes = fit_em_step_bytes
ByteMinGramPPTokenizer.train_prune_bytes = train_prune_bytes

In [15]:
# 1. Create a larger, repetitive text dataset to give the EM engine structures to learn
scale_corpus = [
    "deep learning models are highly dependent on high performance tokenizers",
    "inference engineering requires absolute optimizations at the hardware level",
    "caching key value structures saves massive serving costs during generation steps",
    "compressing sequences into minimal lengths enables ultra low latency inference processing",
    "by tracking flat array indices instead of dictionary pointers we avoid cpu cache misses",
    "we are building this high performance mingram tokenizer step by step in colab"
]
combined_scale_text = " ".join(scale_corpus)

# 2. Mine candidate byte substrings to simulate BPE initialization output
byte_candidates = collections.Counter()
for sentence in scale_corpus:
    words = sentence.split()
    for word in words:
        w_bytes = word.encode('utf-8')
        for start in range(len(w_bytes)):
            for end in range(start + 2, min(start + 10, len(w_bytes) + 1)):
                sub = w_bytes[start:end]
                byte_candidates[sub] += 3

# Inject spaces
byte_candidates[" ".encode('utf-8')] = 50

# 3. Ingest and Train
scale_tokenizer = ByteMinGramPPTokenizer()
scale_tokenizer.load_seed_vocabulary(byte_candidates)

# Shrink the pool down to a tight vocabulary of 300 tokens
scale_tokenizer.train_prune_bytes(scale_corpus, target_vocab_size=300, prune_factor=0.25)

print("\n" + "="*60)
print("--- SCALED HARDFILE INFERENCE BENCHMARK ---")
prod_test_string = "high performance inference processing avoids cpu cache misses"

prod_encoded = scale_tokenizer.encode(prod_test_string)
prod_decoded = scale_tokenizer.decode(prod_encoded)
prod_token_paths = scale_tokenizer.tokenize(prod_test_string)

print(f"Input Text:     '{prod_test_string}'")
print(f"Encoded IDs:    {prod_encoded}")
print(f"Decoded Output: '{prod_decoded}'")
print(f"Tokenizer Path: {prod_token_paths}")
print(f"Validation Check: {prod_test_string == prod_decoded}")

Starting Byte Training. Initial Vocabulary Candidates: 1202
 Pruned 300 byte tokens. Current Size: 902
 Pruned 225 byte tokens. Current Size: 677
 Pruned 169 byte tokens. Current Size: 508
 Pruned 127 byte tokens. Current Size: 381
 Pruned 81 byte tokens. Current Size: 300
Training Complete! Final Clean Matrix Size: 300

--- SCALED HARDFILE INFERENCE BENCHMARK ---
Input Text:     'high performance inference processing avoids cpu cache misses'
Encoded IDs:    [123, 32, 147, 107, 32, 126, 32, 146, 150, 143, 107, 152, 152, 124, 142, 32, 101, 152, 32, 105, 146, 159, 32, 105, 97, 105, 121, 112, 32, 138]
Decoded Output: 'high performance inference processing avoids cpu cache misses'
Tokenizer Path: [b'high', b' ', b'performan', b'ce', b' ', b'inference', b' ', b'p', b'r', b'o', b'ce', b's', b's', b'i', b'ng', b' ', b'avoid', b's', b' ', b'c', b'p', b'u', b' ', b'c', b'a', b'c', b'h', b'e', b' ', b'misses']
Validation Check: True


In [16]:
def run_inference_profile(tokenizer, test_dataset: List[str]):
    """
    Profiles the trained tokenizer, calculating sequence compression
    and total token metrics.
    """
    total_chars = 0
    total_bytes = 0
    total_tokens = 0

    print("--- INFERENCE PROFILE REPORT ---")
    for idx, text in enumerate(test_dataset):
        encoded_ids = tokenizer.encode(text)
        num_tokens = len(encoded_ids)

        total_chars += len(text)
        total_bytes += len(text.encode('utf-8'))
        total_tokens += num_tokens

        # Display the compression profile for individual sentences
        compression_ratio = len(text.encode('utf-8')) / max(1, num_tokens)
        print(f"Sentence {idx+1} | Tokens: {num_tokens} | Bytes/Token: {compression_ratio:.2f}")

    print("-"*40)
    print(f"Total Corpus Bytes:   {total_bytes}")
    print(f"Total Tokens Emitted: {total_tokens}")
    print(f"Global Compression:   {total_bytes / total_tokens:.2f} bytes per token")

# Let's profile using both seen sentences and completely new unseen test text
evaluation_dataset = [
    "high performance inference processing avoids cpu cache misses",
    "deep learning tokenizers reduce latency and serving costs",
    "hardware optimization requires flat array indexing metrics",
    "unseen text containing completely unknown structural variants like matrix multiplication"
]

run_inference_profile(scale_tokenizer, evaluation_dataset)

--- INFERENCE PROFILE REPORT ---
Sentence 1 | Tokens: 30 | Bytes/Token: 2.03
Sentence 2 | Tokens: 30 | Bytes/Token: 1.90
Sentence 3 | Tokens: 38 | Bytes/Token: 1.53
Sentence 4 | Tokens: 84 | Bytes/Token: 1.05
----------------------------------------
Total Corpus Bytes:   264
Total Tokens Emitted: 182
Global Compression:   1.45 bytes per token


In [17]:
def encode_with_special_tokens(self, text: str, allowed_special: Set[str] = None) -> List[int]:
    """
    Production Engine Flag Layer.
    Splits out protected control strings (like <|endoftext|>) via regex
    BEFORE passing text to the main byte tokenizer lattice solver.
    """
    if not text:
        return []

    if allowed_special is None:
        allowed_special = {"<|endoftext|>"}

    # Build a regex pattern targeting the special tokens
    special_pattern = "|".join([re.escape(tok) for tok in allowed_special])
    if not special_pattern:
        return self.encode(text)

    # Split text while keeping matches
    parts = re.split(f"({special_pattern})", text)

    final_ids = []
    for part in parts:
        if part in allowed_special:
            # Direct shortcut map lookup bypassing Viterbi entirely
            final_ids.append(self.token_to_id[part.encode('utf-8')])
        else:
            final_ids.extend(self.encode(part))

    return final_ids

# Inject special tokens safely into the existing scale_tokenizer for testing
special_token = "<|endoftext|>"
special_bytes = special_token.encode('utf-8')

# Rig the system vocabulary maps to accommodate the control token
scale_tokenizer.vocab[special_bytes] = 0.0 # High structural score priority
scale_tokenizer.id_to_token.append(special_bytes)
scale_tokenizer.id_to_token.sort()
scale_tokenizer.token_to_id = {t: i for i, t in enumerate(scale_tokenizer.id_to_token)}
scale_tokenizer.flat_trie = None # Force matrix reset

# Bind the production method
ByteMinGramPPTokenizer.encode_with_special_tokens = encode_with_special_tokens

print("--- PRODUCTION SPECIAL TOKENS LAYER TEST ---")
test_system_prompt = "high performance processing <|endoftext|>"
ids = scale_tokenizer.encode_with_special_tokens(test_system_prompt)
print(f"Input:        '{test_system_prompt}'")
print(f"Assigned IDs: {ids}")
print(f"Decoded back: '{scale_tokenizer.decode(ids)}'")

--- PRODUCTION SPECIAL TOKENS LAYER TEST ---
Input:        'high performance processing <|endoftext|>'
Assigned IDs: [124, 32, 148, 108, 32, 147, 151, 144, 108, 153, 153, 125, 143, 32, 61]
Decoded back: 'high performance processing <|endoftext|>'
